# GPU Pipeline Physics Validation

This notebook validates the physics outputs from the GPU pipeline modules:

1. **numberCountsFull** - 4D integration for cluster number counts N(lambda_ob, z_ob)
2. **bSelMargGPU** - Selection bias marginalisation (P1, I1, J integrals)
3. **ShearPrjGPU** - Projected shear quantities (Sigma_prj, DSigma_prj, gamma_t^prj)

## Physics Overview

The DES Y3 cluster cosmology analysis uses:
- 4 richness bins: [20,30], [30,45], [45,60], [60,200]
- 3 redshift bins: [0.20,0.35], [0.35,0.50], [0.50,0.65]
- Total: 12 bins for number counts

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Style settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 11

# Set output directories - adjust these paths as needed
Y3_DIR = os.environ.get('Y3_CLUSTER_CPP_DIR', '..')
GPU_TEST_DIR = Path(Y3_DIR) / 'cosmosis_tests' / 'gpu'

# Output directories from test runs
NUMBER_COUNTS_OUTPUT = GPU_TEST_DIR / 'number_counts_full_output'
SHEAR_PRJ_OUTPUT = GPU_TEST_DIR / 'shear_prj_gpu_output'
BSEL_MARG_OUTPUT = GPU_TEST_DIR / 'bsel_marg_gpu_output'

print(f"Y3_CLUSTER_CPP_DIR: {Y3_DIR}")
print(f"GPU test directory: {GPU_TEST_DIR}")

---
## 1. Number Counts Validation

The number counts module computes:
$$N(\lambda_{ob}, z_{ob}) = \int d\lambda_{tr} \int dz_t \int d\ln M \int d\lambda_{ob}' \cdot P(\lambda_{ob}'|\lambda_{tr}) \cdot P(\lambda_{tr}|M,z) \cdot P(z_{ob}|z_t) \cdot \frac{dn}{d\ln M} \cdot \frac{dV}{d\Omega dz}$$

### Expected behaviour:
- Higher richness bins should have fewer clusters (HMF falls steeply with mass)
- Number counts should increase with redshift bin width
- Total counts should be O(1000-10000) for DES Y3 footprint

In [ ]:
def load_cosmosis_output(output_dir, section, key):
    """Load a value from CosmoSIS test output directory."""
    filepath = Path(output_dir) / section / f"{key}.txt"
    if filepath.exists():
        return np.loadtxt(filepath)
    else:
        print(f"Warning: {filepath} not found")
        return None

def load_number_counts(output_dir):
    """Load number counts results."""
    vals = load_cosmosis_output(output_dir, 'number_counts_full', 'vals')
    return vals

# Try to load number counts
nc_vals = load_number_counts(NUMBER_COUNTS_OUTPUT)

if nc_vals is not None:
    print(f"Loaded {len(nc_vals)} number count values")
    print(f"Values: {nc_vals}")
else:
    print("Number counts output not found. Run the test first:")
    print("  cosmosis cosmosis_tests/gpu/number_counts_full_test.ini")

In [ ]:
# DES Y3 bin definitions
richness_bins = [(20, 30), (30, 45), (45, 60), (60, 200)]
redshift_bins = [(0.20, 0.35), (0.35, 0.50), (0.50, 0.65)]

richness_labels = [f'$\\lambda \\in [{lo},{hi}]$' for lo, hi in richness_bins]
redshift_labels = [f'$z \\in [{lo:.2f},{hi:.2f}]$' for lo, hi in redshift_bins]

if nc_vals is not None and len(nc_vals) == 12:
    # Reshape to (4 richness, 3 redshift)
    nc_matrix = nc_vals.reshape(4, 3)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Number counts vs richness bin for each z
    ax1 = axes[0]
    richness_centers = [25, 37.5, 52.5, 130]
    for iz, zlabel in enumerate(redshift_labels):
        ax1.semilogy(richness_centers, nc_matrix[:, iz], 'o-', label=zlabel, markersize=8)
    ax1.set_xlabel('Richness bin center $\\lambda$')
    ax1.set_ylabel('Number counts N($\\lambda$, z)')
    ax1.set_title('Number Counts vs Richness')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Heatmap
    ax2 = axes[1]
    im = ax2.imshow(nc_matrix.T, origin='lower', aspect='auto', cmap='viridis')
    ax2.set_xticks(range(4))
    ax2.set_xticklabels([f'{lo}-{hi}' for lo, hi in richness_bins])
    ax2.set_yticks(range(3))
    ax2.set_yticklabels([f'{lo:.2f}-{hi:.2f}' for lo, hi in redshift_bins])
    ax2.set_xlabel('Richness bin')
    ax2.set_ylabel('Redshift bin')
    ax2.set_title('Number Counts Heatmap')
    plt.colorbar(im, ax=ax2, label='N')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n=== Number Counts Summary ===")
    print(f"Total counts: {nc_vals.sum():.2e}")
    print(f"Min: {nc_vals.min():.2e}, Max: {nc_vals.max():.2e}")
    print(f"\nPer richness bin (summed over z):")
    for i, (lo, hi) in enumerate(richness_bins):
        print(f"  lambda=[{lo},{hi}]: {nc_matrix[i,:].sum():.2e}")
else:
    print("Cannot plot - need 12 number count values")

---
## 2. Selection Bias (bSelMargGPU) Validation

The bSelMargGPU module computes the marginalised selection bias integrals:
- **P1**: $\int d\lambda_{tr} \int dz_t \int d\ln M \cdot P(\lambda_{ob}|\lambda_{tr}) \cdot P(\lambda_{tr}|M,z) \cdot ...$
- **I1**: Similar integral with b(M,z) weight
- **J**: Normalisation integral

The selection bias is: $b_{sel} = I1 / P1$

In [ ]:
def load_bsel_marg(output_dir):
    """Load bSelMargGPU results."""
    p1 = load_cosmosis_output(output_dir, 'b_sel_marg', 'p1')
    i1 = load_cosmosis_output(output_dir, 'b_sel_marg', 'i1')
    j = load_cosmosis_output(output_dir, 'b_sel_marg', 'j')
    return p1, i1, j

p1, i1, j = load_bsel_marg(BSEL_MARG_OUTPUT)

if p1 is not None:
    print(f"Loaded bSelMarg results: {len(p1)} grid points")
    print(f"P1 range: [{p1.min():.3e}, {p1.max():.3e}]")
    print(f"I1 range: [{i1.min():.3e}, {i1.max():.3e}]")
    
    # Compute b_sel = I1/P1
    b_sel = i1 / p1
    print(f"\nb_sel = I1/P1 range: [{b_sel.min():.3f}, {b_sel.max():.3f}]")
else:
    print("bSelMarg output not found. Run the test first.")

In [ ]:
if p1 is not None and len(p1) == 12:
    # Reshape to (4 lambda bins, 3 z bins)
    p1_mat = p1.reshape(3, 4).T  # Transpose to get (lambda, z)
    i1_mat = i1.reshape(3, 4).T
    b_sel_mat = i1_mat / p1_mat
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # P1
    ax = axes[0]
    im = ax.imshow(p1_mat.T, origin='lower', aspect='auto', cmap='Blues')
    ax.set_xticks(range(4))
    ax.set_xticklabels(['0', '1', '2', '3'])
    ax.set_yticks(range(3))
    ax.set_yticklabels(['z1', 'z2', 'z3'])
    ax.set_xlabel('Lambda bin')
    ax.set_ylabel('Redshift bin')
    ax.set_title('P1 (normalisation)')
    plt.colorbar(im, ax=ax)
    
    # I1
    ax = axes[1]
    im = ax.imshow(i1_mat.T, origin='lower', aspect='auto', cmap='Oranges')
    ax.set_xticks(range(4))
    ax.set_xticklabels(['0', '1', '2', '3'])
    ax.set_yticks(range(3))
    ax.set_yticklabels(['z1', 'z2', 'z3'])
    ax.set_xlabel('Lambda bin')
    ax.set_title('I1 (bias-weighted)')
    plt.colorbar(im, ax=ax)
    
    # b_sel
    ax = axes[2]
    im = ax.imshow(b_sel_mat.T, origin='lower', aspect='auto', cmap='RdYlGn')
    ax.set_xticks(range(4))
    ax.set_xticklabels(['0', '1', '2', '3'])
    ax.set_yticks(range(3))
    ax.set_yticklabels(['z1', 'z2', 'z3'])
    ax.set_xlabel('Lambda bin')
    ax.set_title('$b_{sel} = I1/P1$')
    plt.colorbar(im, ax=ax)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Selection Bias Summary ===")
    print(f"b_sel by lambda bin (averaged over z):")
    for i in range(4):
        print(f"  bin {i}: {b_sel_mat[i,:].mean():.3f}")

---
## 3. Shear Projection (ShearPrjGPU) Validation

The ShearPrjGPU module computes projected surface density profiles:
- **Sigma_prj**: Projected surface mass density $\Sigma(R)$
- **DSigma_prj**: Differential surface density $\Delta\Sigma(R) = \bar{\Sigma}(<R) - \Sigma(R)$
- **gamma_t^prj**: Tangential shear $\gamma_t = \Delta\Sigma \cdot \Sigma_{crit}^{-1}$

Each has random (1-halo) and clustering (2-halo) components.

In [ ]:
def load_shear_prj(output_dir):
    """Load ShearPrjGPU results."""
    results = {}
    for section in ['sigma_prj', 'dsigma_prj', 'shear_prj']:
        results[section] = {}
        for key in ['vals', 'rnd', 'cl']:
            results[section][key] = load_cosmosis_output(output_dir, section, key)
    return results

shear_results = load_shear_prj(SHEAR_PRJ_OUTPUT)

if shear_results['sigma_prj']['vals'] is not None:
    sigma_vals = shear_results['sigma_prj']['vals']
    print(f"Loaded shear results: {len(sigma_vals)} grid points")
    print(f"\nSigma_prj range: [{sigma_vals.min():.3e}, {sigma_vals.max():.3e}]")
    print(f"DSigma_prj range: [{shear_results['dsigma_prj']['vals'].min():.3e}, {shear_results['dsigma_prj']['vals'].max():.3e}]")
    print(f"gamma_t range: [{shear_results['shear_prj']['vals'].min():.3e}, {shear_results['shear_prj']['vals'].max():.3e}]")
else:
    print("Shear output not found. Run the test first:")
    print("  cosmosis cosmosis_tests/gpu/shear_prj_test.ini")

In [ ]:
if shear_results['sigma_prj']['vals'] is not None:
    # The grid is: 3 z bins x 4 lambda bins x 3 R values = 36 points
    # Ordering: lambda varies fastest, then R, then z
    n_points = len(shear_results['sigma_prj']['vals'])
    
    # Radii used in the test
    radii = [0.5, 2.0, 5.0]  # Mpc/h
    n_r = len(radii)
    n_lambda = 4
    n_z = 3
    
    if n_points == n_r * n_lambda * n_z:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        
        # Extract data for lambda bin 0, all z and R
        for section, ax, title in [
            ('sigma_prj', axes[0,0], '$\\Sigma_{prj}(R)$'),
            ('dsigma_prj', axes[0,1], '$\\Delta\\Sigma_{prj}(R)$'),
            ('shear_prj', axes[1,0], '$\\gamma_t^{prj}(R)$')
        ]:
            vals = shear_results[section]['vals']
            rnd = shear_results[section]['rnd']
            cl = shear_results[section]['cl']
            
            # Plot for lambda bin 1 (index 1), each z bin
            for iz in range(n_z):
                # Indices: iz * (n_lambda * n_r) + ilam * n_r + ir
                ilam = 1
                indices = [iz * (n_lambda * n_r) + ilam * n_r + ir for ir in range(n_r)]
                
                ax.loglog(radii, [vals[i] for i in indices], 'o-', 
                         label=f'z bin {iz}', markersize=8)
            
            ax.set_xlabel('R [Mpc/h]')
            ax.set_ylabel(title)
            ax.set_title(f'{title} ($\\lambda$ bin 1)')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # Random vs Clustering decomposition
        ax = axes[1,1]
        vals = shear_results['dsigma_prj']['vals']
        rnd = shear_results['dsigma_prj']['rnd']
        cl = shear_results['dsigma_prj']['cl']
        
        # For z bin 1, lambda bin 1
        iz, ilam = 1, 1
        indices = [iz * (n_lambda * n_r) + ilam * n_r + ir for ir in range(n_r)]
        
        ax.loglog(radii, [vals[i] for i in indices], 'ko-', label='Total', markersize=10)
        ax.loglog(radii, [rnd[i] for i in indices], 'b^--', label='Random (1h)', markersize=8)
        ax.loglog(radii, [cl[i] for i in indices], 'rs--', label='Clustering (2h)', markersize=8)
        
        ax.set_xlabel('R [Mpc/h]')
        ax.set_ylabel('$\\Delta\\Sigma$')
        ax.set_title('$\\Delta\\Sigma$ decomposition (z=1, $\\lambda$=1)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print(f"Unexpected number of points: {n_points} (expected {n_r * n_lambda * n_z})")

---
## 4. Physics Consistency Checks

Verify that the GPU results satisfy expected physical constraints.

In [ ]:
print("=== Physics Consistency Checks ===")
print()

# Check 1: Number counts decrease with richness
if nc_vals is not None and len(nc_vals) == 12:
    nc_matrix = nc_vals.reshape(4, 3)
    nc_by_richness = nc_matrix.sum(axis=1)  # Sum over z
    
    monotonic = all(nc_by_richness[i] >= nc_by_richness[i+1] for i in range(3))
    status = "PASS" if monotonic else "FAIL"
    print(f"[{status}] Number counts decrease with richness")
    if not monotonic:
        print(f"       Values: {nc_by_richness}")

# Check 2: Selection bias is O(1) and positive
if p1 is not None:
    b_sel = i1 / p1
    reasonable = (b_sel > 0).all() and (b_sel < 10).all()
    status = "PASS" if reasonable else "FAIL"
    print(f"[{status}] Selection bias b_sel in reasonable range (0, 10)")
    print(f"       Range: [{b_sel.min():.3f}, {b_sel.max():.3f}]")

# Check 3: DSigma positive and decreasing with R
if shear_results['dsigma_prj']['vals'] is not None:
    dsig = shear_results['dsigma_prj']['vals']
    positive = (dsig > 0).all()
    status = "PASS" if positive else "FAIL"
    print(f"[{status}] DSigma_prj is positive everywhere")

# Check 4: gamma_t positive
if shear_results['shear_prj']['vals'] is not None:
    gamma = shear_results['shear_prj']['vals']
    positive = (gamma > 0).all()
    status = "PASS" if positive else "FAIL"
    print(f"[{status}] gamma_t^prj is positive everywhere")

print("\n=== End of Validation ===")

---
## 5. CPU vs GPU Comparison (Optional)

If CPU reference values are available, compare GPU outputs against them.

In [ ]:
# Placeholder for CPU vs GPU comparison
# Add CPU reference data paths here when available

CPU_REFERENCE_DIR = None  # Set to path with CPU outputs

if CPU_REFERENCE_DIR:
    print("Loading CPU reference values...")
    # Compare and compute relative differences
else:
    print("No CPU reference data configured.")
    print("To enable comparison, set CPU_REFERENCE_DIR to a directory containing:")
    print("  - number_counts_full/vals.txt")
    print("  - b_sel_marg/p1.txt, i1.txt, j.txt")
    print("  - sigma_prj/vals.txt, dsigma_prj/vals.txt, shear_prj/vals.txt")